# Test: `load_fed_curves`

Verifies that `treasury_bonds.py` downloads, parses, and saves the Fed GSW dataset correctly.

In [ ]:
import sys
sys.path.insert(0, "../src")

from termstructure.ingest.treasury_bonds import load_fed_curves, OUTPUT_PATH

## Check 1 — Download and inspect

In [ ]:
df = load_fed_curves()

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nDate range:", df.index.min(), "→", df.index.max())
df.head()

## Check 2 — Svensson parameters look reasonable

beta0 is the long-run asymptotic rate. The data goes back to 1961 and covers the
early-1980s rate spike when US yields hit ~16%, so beta0 can legitimately exceed 20%.
We just check it's finite and positive. tau1 must always be positive.

In [ ]:
params = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]
print("Svensson parameter stats:")
print(df[params].describe().round(3))

assert (df["beta0"].dropna() > 0).all(), "beta0 should be positive"
assert (df["tau1"].dropna() > 0).all(), "tau1 should be positive"
print(f"\nbeta0 range: {df['beta0'].min():.2f}% → {df['beta0'].max():.2f}%")
print("Parameter sanity checks passed.")

## Check 3 — Zero yield panel is complete

sveny01–sveny30 should have very few NaNs (only on holidays).
Yields should be in a plausible range (0–20%).

In [ ]:
zero_cols = [f"sveny{i:02d}" for i in range(1, 31)]
zero_panel = df[zero_cols]

nan_per_col = zero_panel.isna().mean().round(3)
print("NaN rate per column:")
print(nan_per_col.to_string())

ymin = zero_panel.min().min()
ymax = zero_panel.max().max()
print(f"\nYield range: {ymin:.2f}% → {ymax:.2f}%")

## Check 4 — Spot-check against known Fed values

On 2023-01-03, the Fed's published beta0 should be around 4.0–4.5
and the 10-year zero yield (sveny10) should be close to the CMT DGS10 value of 3.79%.

In [ ]:
row = df.loc["2023-01-03", ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2", "sveny10"]]
print(row)

sveny10 = row["sveny10"]
assert abs(sveny10 - 3.79) < 0.2, f"sveny10 on 2023-01-03 expected ~3.79%, got {sveny10:.2f}%"
print(f"\nSpot check passed: 10Y zero yield on 2023-01-03 = {sveny10:.2f}%")

## Check 5 — Parquet round-trip

In [ ]:
import pandas as pd

assert OUTPUT_PATH.exists(), f"Parquet not found at {OUTPUT_PATH}"
df_disk = pd.read_parquet(OUTPUT_PATH)

print("Saving to:", OUTPUT_PATH)
print("In-memory shape:", df.shape)
print("On-disk shape:  ", df_disk.shape)

assert df_disk.shape == (len(df), len(df.columns) + 1), "Unexpected shape on disk"
print("\nRound-trip check passed.")